In [1]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

2024-10-25 09:33:17.172115: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-25 09:33:17.240992: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-25 09:33:17.262090: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-25 09:33:17.371778: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-25 09:33:19.298845: W tensorflow/compiler/tf2

Num GPUs Available:  1


I0000 00:00:1729829001.412400    6406 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1729829001.571233    6406 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1729829001.571744    6406 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355


In [2]:
import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        print("Found a GPU with the name:", gpu)
else:
    print("Failed to detect a GPU.")


Found a GPU with the name: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [3]:
import cv2
import numpy as np
import os

def load_images_from_folder(folder, label, img_size=(224, 224)):
    images = []
    labels = []
    for filename in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, filename))
        if img is not None:
            img = cv2.resize(img, img_size)  # Resize image to target size
            images.append(img)
            labels.append(label)
    return images, labels

# Load images from '0' folder (without human) and '1' folder (with human)
no_human_images, no_human_labels = load_images_from_folder('/mnt/A42AC5272AC4F778/Kongsberg/human detection dataset/0', 0)
with_human_images, with_human_labels = load_images_from_folder('/mnt/A42AC5272AC4F778/Kongsberg/human detection dataset/1', 1)

# Combine the data
images = np.array(no_human_images + with_human_images)
labels = np.array(no_human_labels + with_human_labels)

# Normalize the images to be between 0 and 1
images = images / 255.0


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known inc

In [4]:
from sklearn.utils import shuffle

images, labels = shuffle(images, labels, random_state=42)


In [5]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

# Load MobileNetV2 as a base model with pre-trained weights
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Add new layers for classification
x = base_model.output
x = GlobalAveragePooling2D()(x)  # Add pooling layer
x = Dense(512, activation='relu')(x)  # Add fully connected layer
predictions = Dense(1, activation='sigmoid')(x)  # Output layer for binary classification

# Create a new model
model = Model(inputs=base_model.input, outputs=predictions)

# Freeze the base model layers
for layer in base_model.layers:
    layer.trainable = False

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


I0000 00:00:1729829011.396666    6406 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1729829011.396944    6406 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1729829011.397179    6406 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1729829011.476666    6406 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

In [6]:
# Train the model
history = model.fit(images, labels, epochs=10, batch_size=32, validation_split=0.2)


Epoch 1/10


I0000 00:00:1729829016.297946    6674 service.cc:146] XLA service 0x77cfd4110880 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1729829016.297993    6674 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2024-10-25 09:33:36.366010: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-10-25 09:33:36.822731: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8907


 3/23 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.4583 - loss: 1.2167

I0000 00:00:1729829022.896519    6674 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


23/23 ━━━━━━━━━━━━━━━━━━━━ 17s 359ms/step - accuracy: 0.6514 - loss: 0.9550 - val_accuracy: 0.8270 - val_loss: 0.3906
Epoch 2/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.9045 - loss: 0.2601 - val_accuracy: 0.8865 - val_loss: 0.3077
Epoch 3/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.9548 - loss: 0.1559 - val_accuracy: 0.9135 - val_loss: 0.3034
Epoch 4/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.9832 - loss: 0.1069 - val_accuracy: 0.8973 - val_loss: 0.2968
Epoch 5/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.9805 - loss: 0.0781 - val_accuracy: 0.9027 - val_loss: 0.3008
Epoch 6/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.9833 - loss: 0.0603 - val_accuracy: 0.9189 - val_loss: 0.3345
Epoch 7/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.9925 - loss: 0.0487 - val_accuracy: 0.8486 - val_loss: 0.4168
Epoch 8/10
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.9895 - loss: 0.0592 - val_accuracy: 0.9135 - val_loss: 

In [7]:
# Evaluate the model on the validation data
val_loss, val_accuracy = model.evaluate(images, labels)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_accuracy}")


29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.9964 - loss: 0.0214
Validation Loss: 0.08686504513025284
Validation Accuracy: 0.9815418124198914


In [8]:
# Save the model
model.save('human_detection_model.h5')


In [9]:
from tensorflow.keras.models import load_model

# Load the trained model
model = load_model('human_detection_model.h5')


In [10]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
import time

# Load the model
model = load_model('human_detection_model.h5')

# Initialize the video capture (replace '0' with your CCTV video source or file path)
cap = cv2.VideoCapture('/mnt/A42AC5272AC4F778/Kongsberg/office_video.mp4')  # Use '0' for webcam or path to video file

# Countdown duration in seconds
countdown_duration = 60
shutdown_display_duration = 15
last_empty_time = time.time()
is_countdown_active = False
shutdown_message_displayed = False
shutdown_start_time = None

# Thresholds for consecutive frame confirmation
occupancy_threshold = 3  # Number of consecutive frames to confirm occupancy
empty_threshold = 3  # Number of consecutive frames to confirm emptiness
occupancy_count = 0
empty_count = 0

def control_lights_and_hvac(turn_off):
    if turn_off:
        # Replace these print statements with actual commands to control your lights and HVAC
        print("Turning off lights and HVAC.")
    else:
        # Replace these print statements with actual commands to turn on lights and HVAC if needed
        print("Turning on lights and HVAC.")

# Check if video capture is opened
if not cap.isOpened():
    print("Error: Unable to open video source.")
    exit()

while True:
    ret, frame = cap.read()
    
    if not ret:
        # Exit the loop if no more frames are available
        print("No more frames or error in capturing video.")
        break

    # Preprocess the frame
    img = cv2.resize(frame, (224, 224))
    img = np.expand_dims(img, axis=0)  # Add batch dimension
    img = img / 255.0  # Normalize

    # Predict
    prediction = model.predict(img)
    is_occupied = prediction[0][0] > 0.7  # Adjust threshold to reduce false positives

    # Initialize default values for label and color
    label = 'Processing...'
    color = (255, 255, 255)  # White color as default

    if is_occupied:
        # Increase occupancy count, reset empty count
        occupancy_count += 1
        empty_count = 0
        if occupancy_count >= occupancy_threshold:
            # Room is officially occupied after consecutive frames
            last_empty_time = time.time()
            is_countdown_active = False
            shutdown_message_displayed = False
            shutdown_start_time = None
            label = 'Occupied'
            color = (0, 255, 0)  # Green color for occupied
    else:
        # Increase empty count, reset occupancy count
        empty_count += 1
        occupancy_count = 0
        if empty_count >= empty_threshold:
            # Room is officially empty after consecutive frames
            if not is_countdown_active:
                is_countdown_active = True
                last_empty_time = time.time()

            # Calculate the remaining countdown time
            elapsed_time = time.time() - last_empty_time
            remaining_time = max(countdown_duration - elapsed_time, 0)

            if remaining_time <= 0:
                if not shutdown_message_displayed:
                    shutdown_start_time = time.time()
                    control_lights_and_hvac(turn_off=True)  # Command to turn off lights and HVAC
                    shutdown_message_displayed = True

                # Check if the shutdown message should still be displayed
                if shutdown_start_time and time.time() - shutdown_start_time <= shutdown_display_duration:
                    label = 'Lights and HVAC system shutting down...'
                    color = (0, 0, 255)  # Red color for shutdown
                else:
                    break
            else:
                label = f'Empty - Countdown: {int(remaining_time)}s'
                color = (0, 0, 255)  # Red color for empty countdown

    # Display result on the frame
    cv2.putText(frame, label, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 2)

    # Show the frame
    cv2.imshow('Room Occupancy', frame)

    # Break the loop on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the capture and close windows
cap.release()
cv2.destroyAllWindows()


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━

KeyboardInterrupt: 